<a href="https://colab.research.google.com/github/gabscorreia/gcp-data-pipeline-serverless/blob/main/camada_gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas_gbq
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

In [4]:
print("Carregando as tabelas limpas da camada Silver...")

# Definindo o ID do projeto que estava faltando
project_id = "strong-planet-484414-a0"

# 1. Lendo as tabelas principais necessárias para o cruzamento de negócio
orders = pandas_gbq.read_gbq(
    f"SELECT * FROM `{project_id}.ecommerce_silver.tb_orders_`",
    project_id=project_id,
    location="southamerica-east1",
)
customers = pandas_gbq.read_gbq(
    f"SELECT * FROM `{project_id}.ecommerce_silver.tb_cliente`",
    project_id=project_id,
    location="southamerica-east1",
)
order_items = pandas_gbq.read_gbq(
    f"SELECT * FROM `{project_id}.ecommerce_silver.tb_order_items`",
    project_id=project_id,
    location="southamerica-east1",
)
products = pandas_gbq.read_gbq(
    f"SELECT * FROM `{project_id}.ecommerce_silver.tb_products_`",
    project_id=project_id,
    location="southamerica-east1",
)
payments = pandas_gbq.read_gbq(
    f"SELECT * FROM `{project_id}.ecommerce_silver.tb_order_payments`",
    project_id=project_id,
    location="southamerica-east1",
)

print("Realizando os cruzamentos (joins) para formar a tabela ampla...")

Carregando as tabelas limpas da camada Silver...
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Realizando os cruzamentos (joins) para formar a tabela ampla...


In [5]:
# 2. Fazendo merges sucessivos para juntar contexto de cliente, itens, produtos e pagamentos ao pedido
df_gold = (
    orders.merge(customers, on="customer_id", how="inner")
    .merge(order_items, on="order_id", how="inner")
    .merge(products, on="product_id", how="left")
    .merge(payments, on="order_id", how="left")
)

In [6]:
# 3. Seleciona e organiza apenas as colunas mais relevantes para análise de negócio
colunas_interesse = [
    "order_id",
    "customer_id",
    "customer_city",
    "customer_state",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "product_id",
    "product_category_name",
    "price",
    "freight_value",
    "payment_type",
    "payment_installments",
    "payment_value",
]

In [7]:
# Mantém apenas as colunas que existem no DataFrame resultante
df_gold = df_gold[
    [col for col in colunas_interesse if col in df_gold.columns]
].copy()

print(f" -> Total de linhas na tabela consolidada Gold: {len(df_gold)}")

 -> Total de linhas na tabela consolidada Gold: 117604


In [8]:
# 4. Grava a tabela final na Camada Gold no BigQuery
dataset_gold = "ecommerce_gold"
tabela_gold = "mart_vendas"

print(f"A guardar a tabela '{tabela_gold}' na camada Gold...")
df_gold.to_gbq(
    destination_table=f"{dataset_gold}.{tabela_gold}",
    project_id=project_id,
    if_exists="replace",
    location="southamerica-east1",
)

print(
    "\n=== CAMADA GOLD CONCLUÍDA ==="
)

A guardar a tabela 'mart_vendas' na camada Gold...


/tmp/ipykernel_402/2745350613.py:6: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_gold.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 7884.03it/s]


=== CAMADA GOLD CONCLUÍDA ===
